# Analyzing some tendencies after slim dimensionality reduction with Random Embeddings

In this script we just focus on some partial dataset wherein we use $d=20$ on just a subset of functions from BBOB (i.e. $f=\{1,8,11,16,20 \}$). What we want to do is then check how some features from the Exploratory Landscape Analysis Toolbox (i.e. FLACCO) behave under a projection with Random Embeddings with a projection matrix $P \in \mathbb{R}^{d \times q}$. In this case we set $q=10$ as a preliminary value to have $50\%$ reduction.

## Procedure
For this case, we use the package `sklearn.random_projection.GaussianRandomProjection` in order to generate the random projection matrices. This generates random matrices where $\left\|P \right\|_{2}=1.0$. With this in mind, we generate 10 different random projection matrices of this kind. Furthermore, we generated beforehand initial 10 datasets of $\mathbf{X} \in [-5,5]^{d}$ with $100 d$ samples via `LHS`. *A priori*, the sampling scheme wouldn't signify a very different performance with respect to other sampling available techniques since the Random Projection will transform any initial space-filling optimized Design of Experiments scheme into some degenerate one since there'll be artificial correlations caused by the projection into the reduced latent space.

### Sampled ELA Features
After what was discussed in some references (i.e. *Expressiveness and Robustness of Landscape Features*, Renau et al., 2019), we sampled the following sets of ELA Features:
- **Dispersion (disp)**
- **Information Content (ic)**
- **Nearest Best Clustering (nbc)**
- **Principal Component Analysis (pca)**
- **Meta Model (ela_meta)**
- **$y$-distribution (ela_distr)**
- **Level Set (ela_level)**

From the set of features, it's self-evident that the **$y$-distribution** features, these are invariant to any dimensionality reduction techniques since the scope of these features is just the *objective space*. However to account for subsampling in a high dimensional space we proceed to generate subsets of $100 q$ by sampling without replacement from the initial dataset. The idea is then two-fold: 1) We simulate what would happen what happens by keeping the same *dimensionality propertion* and 2) we can study the stability and possible distributions of the features. We hypothesize that the distributions of features by performing this sampling will depend primarily on the sampled function.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
# Import necessary libraries
import pandas as pd
import numpy as np
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Load the complete data

complete_data = pd.read_csv('complete_data_2.csv')

# Display the first few rows of the dataframe
complete_data

In [ ]:
# Just use functions 1, 8, 11, 16, 20

sets_of_functions = {1, 8, 11, 16, 20}

# Slice the dataframe to only include these functions
filtered_data = complete_data[complete_data['function_idx'].isin(sets_of_functions)]

In [ ]:
filtered_data

In [ ]:
# Now load the reduced data
reduced_data_05 = pd.read_parquet('reduced_1_200_0.5.parquet')
reduced_data_025 = pd.read_parquet('reduced_1_200_0.25.parquet')

In [ ]:
reduced_data_05

In [ ]:
reduced_data_025

Now that the reduced data is loaded, we proceed with the analysis and plotting as needed. Now we proceed to remove `runtime` features from both datasets.

In [ ]:
# Remove columns where names contain `runtime`

cols_to_remove = [col for col in reduced_data_05.columns if 'runtime' in col]

reduced_data_05_cleaned = reduced_data_05.drop(columns=cols_to_remove)

reduced_data_05_cleaned

In [ ]:
# Remove columns where names contain `runtime`

cols_to_remove = [col for col in reduced_data_025.columns if 'runtime' in col]

reduced_data_025_cleaned = reduced_data_025.drop(columns=cols_to_remove)

reduced_data_025_cleaned

In [ ]:
cols_to_remove = [col for col in complete_data.columns if 'runtime' in col]

complete_data_cleaned = complete_data.drop(columns=cols_to_remove)

complete_data_cleaned

In [ ]:
# These are the prefixes of the feature sets
feature_set_prefixes = (
    'ela_level',
    'ela_meta',
    'ela_distr',
    'nbc',
    'pca',
    'ic',
    'disp',
    'fitness_distance'
)

In [ ]:
def plot_feature_contrast_per_fid_instance_original_database(
    original_database: pd.DataFrame,
    reduced_database: pd.DataFrame,
    feature_name: str,
    n_samples: int = 2000,
    dataset_seed: int = 123,
    fid: int = 1,
    instance: int = 0,
    dimension: int = 20,
):
    # Filter original database (single realization)
    original_filtered = original_database[
        (original_database["function_idx"] == fid)
        & (original_database["instance_idx"] == instance)
        & (original_database["dimension"] == dimension)
        & (original_database["n_samples"] == n_samples)
        & (original_database["seed_lhs"] == dataset_seed)
    ]

    # Filter reduced database (distribution over embedding_seed)
    reduced_filtered = reduced_database[
        (reduced_database["function_idx"] == fid)
        & (reduced_database["instance_idx"] == instance)
        & (reduced_database["dimension"] == dimension)
        & (reduced_database["n_samples"] == n_samples)
        & (reduced_database["seed_lhs"] == dataset_seed)
    ]

    # Extract feature(s)
    original_feature = original_filtered.loc[
        :, original_filtered.columns.str.contains(feature_name)
    ]

    reduced_feature = reduced_filtered.loc[
        :, reduced_filtered.columns.str.contains(feature_name)
    ]

    if original_feature.shape[0] != 1:
        raise ValueError("Original feature must contain exactly one row.")

    # --- Long-format reduced data with embedding seed ---
    reduced_long = reduced_filtered[["embedding_seed"]].join(reduced_feature)
    reduced_long = reduced_long.melt(
        id_vars="embedding_seed",
        var_name="feature",
        value_name="value",
    )

    # --- Plot ---
    fig, ax = plt.subplots(figsize=(14, 6))

    sns.violinplot(
        data=reduced_long,
        x="embedding_seed",
        y="value",
        inner="quartile",
        cut=0,
        density_norm='width',
        ax=ax,
        color="lightblue",
    )

    # --- Reference line from original feature ---
    for col in original_feature.columns:
        ax.axhline(
            original_feature.iloc[0][col],
            color="darkgreen",
            linestyle="--",
            linewidth=2,
            label="Full Dimensional Feature Value",
        )

    ax.set_title(
        f"{feature_name} | fid={fid}, instance={instance}, dim={dimension}, seed_lhs={dataset_seed}, n_samples={n_samples}"
    )
    ax.set_xlabel("Projection Identifier (embedding seed)")
    ax.set_ylabel(feature_name)

    # Avoid duplicate legend entries
    handles, labels = ax.get_legend_handles_labels()
    if labels:
        ax.legend(handles[:1], labels[:1])

    plt.show()

In [ ]:
# plot_feature_contrast_per_fid_instance_original_database(

plot_feature_contrast_per_fid_instance_original_database(original_database=complete_data_cleaned,
                                                        reduced_database=reduced_data_05_cleaned,
                                                        feature_name='ela_meta.lin_simple.intercept',
                                                        fid=1,
                                                        instance=0,
                                                        n_samples=200,
                                                        dataset_seed=1001)

In [ ]:
plot_feature_contrast_per_fid_instance_original_database(original_database=complete_data_cleaned,
                                                        reduced_database=reduced_data_025_cleaned,
                                                        feature_name='ela_meta.lin_simple.intercept',
                                                        fid=1,
                                                        instance=0,
                                                        n_samples=200,
                                                        dataset_seed=1001)

In [ ]:
plot_feature_contrast_per_fid_instance_original_database(original_database=complete_data_cleaned,
                                                        reduced_database=reduced_data_05_cleaned,
                                                        feature_name='ela_meta.lin_simple.intercept',
                                                        fid=11,
                                                        instance=0,
                                                        n_samples=200,
                                                        dataset_seed=1001)

In [ ]:
plot_feature_contrast_per_fid_instance_original_database(original_database=complete_data_cleaned,
                                                        reduced_database=reduced_data_025_cleaned,
                                                        feature_name='ela_meta.lin_simple.intercept',
                                                        fid=11,
                                                        instance=0,
                                                        n_samples=200,
                                                        dataset_seed=1001)

In [ ]:
plot_feature_contrast_per_fid_instance_original_database(original_database=complete_data_cleaned,
                                                        reduced_database=reduced_data_05_cleaned,
                                                        feature_name='pca.expl_var_PC1.cor_x',
                                                        fid=11,
                                                        instance=0,
                                                        n_samples=200,
                                                        dataset_seed=1001)

In [ ]:
plot_feature_contrast_per_fid_instance_original_database(original_database=complete_data_cleaned,
                                                        reduced_database=reduced_data_025_cleaned,
                                                        feature_name='pca.expl_var_PC1.cor_x',
                                                        fid=11,
                                                        instance=0,
                                                        n_samples=200,
                                                        dataset_seed=1001)

In [ ]:
plot_feature_contrast_per_fid_instance_original_database(original_database=complete_data_cleaned,
                                                        reduced_database=reduced_data_05_cleaned,
                                                        feature_name='ela_meta.lin_simple.adj_r2',
                                                        fid=11,
                                                        instance=0,
                                                        n_samples=200,
                                                        dataset_seed=1001)

In [ ]:
plot_feature_contrast_per_fid_instance_original_database(original_database=complete_data_cleaned,
                                                        reduced_database=reduced_data_025_cleaned,
                                                        feature_name='ela_meta.lin_simple.adj_r2',
                                                        fid=11,
                                                        instance=0,
                                                        n_samples=200,
                                                        dataset_seed=1001)

In [ ]:
# plot_feature_contrast_per_fid_instance_original_database(

plot_feature_contrast_per_fid_instance_original_database(original_database=complete_data_cleaned,
                                                        reduced_database=reduced_data_cleaned,
                                                        feature_name='ela_distr.skewness',
                                                        fid=11,
                                                        instance=0)

In [ ]:
plot_feature_contrast_per_fid_instance_original_database(original_database=complete_data_cleaned,
                                                        reduced_database=reduced_data_cleaned,
                                                        feature_name='ic.h_max',
                                                        fid=8,
                                                        instance=0)

In [ ]:
plot_feature_contrast_per_fid_instance_original_database(original_database=complete_data_cleaned,
                                                        reduced_database=reduced_data_cleaned,
                                                        feature_name='ic.h_max',
                                                        fid=11,
                                                        instance=0)

In [ ]:
plot_feature_contrast_per_fid_instance_original_database(original_database=complete_data_cleaned,
                                                        reduced_database=reduced_data_cleaned,
                                                        feature_name='ic.h_max',
                                                        fid=1,
                                                        instance=0)

In [ ]:
plot_feature_contrast_per_fid_instance_original_database(original_database=complete_data_cleaned,
                                                        reduced_database=reduced_data_cleaned,
                                                        feature_name='ic.eps_s',
                                                        fid=8,
                                                        instance=0)

In [ ]:
plot_feature_contrast_per_fid_instance_original_database(original_database=complete_data_cleaned,
                                                        reduced_database=reduced_data_cleaned,
                                                        feature_name='ic.eps_s',
                                                        fid=1,
                                                        instance=10)

In [ ]:
plot_feature_contrast_per_fid_instance_original_database(original_database=complete_data_cleaned,
                                                        reduced_database=reduced_data_cleaned,
                                                        feature_name='ic.eps_s',
                                                        fid=1,
                                                        instance=2,
                                                        dataset_seed=123)

In [ ]:
plot_feature_contrast_per_fid_instance_original_database(original_database=complete_data_cleaned,
                                                        reduced_database=reduced_data_cleaned,
                                                        feature_name='ic.eps_s',
                                                        fid=20,
                                                        instance=0)

In [ ]:
plot_feature_contrast_per_fid_instance_original_database(original_database=complete_data_cleaned,
                                                        reduced_database=reduced_data_cleaned,
                                                        feature_name='ic.eps_s',
                                                        dataset_seed=124,
                                                        fid=20,
                                                        instance=0)

In [ ]:
plot_feature_contrast_per_fid_instance_original_database(original_database=complete_data_cleaned,
                                                        reduced_database=reduced_data_cleaned,
                                                        feature_name='ela_meta.lin_simple.adj_r2',
                                                        fid=1,
                                                        instance=0,
                                                        dataset_seed=123)

In [ ]:
plot_feature_contrast_per_fid_instance_original_database(original_database=complete_data_cleaned,
                                                        reduced_database=reduced_data_cleaned,
                                                        feature_name='ela_meta.lin_simple.adj_r2',
                                                        fid=1,
                                                        dataset_seed=124)

In [ ]:
plot_feature_contrast_per_fid_instance_original_database(original_database=complete_data_cleaned,
                                                        reduced_database=reduced_data_cleaned,
                                                        feature_name='ela_level.mmce_lda_25',
                                                        fid=1,
                                                        instance=1)

In [ ]:
plot_feature_contrast_per_fid_instance_original_database(original_database=complete_data_cleaned,
                                                        reduced_database=reduced_data_cleaned,
                                                        feature_name='ela_level.mmce_lda_25',
                                                        fid=1,
                                                        instance=2)

In [ ]:
plot_feature_contrast_per_fid_instance_original_database(original_database=complete_data_cleaned,
                                                        reduced_database=reduced_data_cleaned,
                                                        feature_name='ela_level.mmce_lda_25',
                                                        fid=1,
                                                        instance=3)

In [ ]:
plot_feature_contrast_per_fid_instance_original_database(original_database=complete_data_cleaned,
                                                        reduced_database=reduced_data_cleaned,
                                                        feature_name='nbc.nb_fitness.cor',
                                                        fid=1,
                                                        instance=0)

In [ ]:
plot_feature_contrast_per_fid_instance_original_database(original_database=complete_data_cleaned,
                                                        reduced_database=reduced_data_cleaned,
                                                        feature_name='nbc.nb_fitness.cor',
                                                        fid=1,
                                                        dataset_seed=124,
                                                        instance=0)

In [ ]:
plot_feature_contrast_per_fid_instance_original_database(original_database=complete_data_cleaned,
                                                        reduced_database=reduced_data_cleaned,
                                                        feature_name='nbc.nb_fitness.cor',
                                                        fid=1,
                                                        dataset_seed=124,
                                                        instance=1)

In [ ]:
plot_feature_contrast_per_fid_instance_original_database(original_database=complete_data_cleaned,
                                                        reduced_database=reduced_data_cleaned,
                                                        feature_name='ela_meta.lin_simple.adj_r2',
                                                        fid=20,
                                                        dataset_seed=123,
                                                        instance=0)

In [ ]:
plot_feature_contrast_per_fid_instance_original_database(original_database=complete_data_cleaned,
                                                        reduced_database=reduced_data_cleaned,
                                                        feature_name='ela_meta.lin_simple.adj_r2',
                                                        fid=20,
                                                        dataset_seed=124,
                                                        instance=0)

In [ ]:
plot_feature_contrast_per_fid_instance_original_database(original_database=complete_data_cleaned,
                                                        reduced_database=reduced_data_cleaned,
                                                        feature_name='ela_meta.lin_simple.adj_r2',
                                                        fid=8,
                                                        dataset_seed=123,
                                                        instance=0)

In [ ]:
plot_feature_contrast_per_fid_instance_original_database(original_database=complete_data_cleaned,
                                                        reduced_database=reduced_data_cleaned,
                                                        feature_name='ela_meta.lin_simple.adj_r2',
                                                        fid=8,
                                                        dataset_seed=124,
                                                        instance=0)

In [ ]:
plot_feature_contrast_per_fid_instance_original_database(original_database=complete_data_cleaned,
                                                        reduced_database=reduced_data_cleaned,
                                                        feature_name='pca.expl_var_PC1.cov_x',
                                                        fid=20,
                                                        dataset_seed=123,
                                                        instance=0)

In [ ]:
plot_feature_contrast_per_fid_instance_original_database(original_database=complete_data_cleaned,
                                                        reduced_database=reduced_data_cleaned,
                                                        feature_name='pca.expl_var_PC1.cov_x',
                                                        fid=20,
                                                        dataset_seed=124,
                                                        instance=0)

In [ ]:
plot_feature_contrast_per_fid_instance_original_database(original_database=complete_data_cleaned,
                                                        reduced_database=reduced_data_cleaned,
                                                        feature_name='pca.expl_var_PC1.cov_x',
                                                        fid=1,
                                                        dataset_seed=123,
                                                        instance=0)

In [ ]:
plot_feature_contrast_per_fid_instance_original_database(original_database=complete_data_cleaned,
                                                        reduced_database=reduced_data_cleaned,
                                                        feature_name='pca.expl_var_PC1.cov_x',
                                                        fid=1,
                                                        dataset_seed=124,
                                                        instance=0)

In [ ]:
plot_feature_contrast_per_fid_instance_original_database(original_database=complete_data_cleaned,
                                                        reduced_database=reduced_data_cleaned,
                                                        feature_name='pca.expl_var_PC1.cov_x',
                                                        fid=20,
                                                        dataset_seed=124,
                                                        instance=0)